# Notebook 04b — Phase 2: MobileFaceNet **from scratch** on Bollywood Faces

**Phase:** 2 (Indian demographic specialization, from-scratch branch)  
**Purpose:** Build MobileFaceNet architecture in Keras with random weights, then train end-to-end on the 100-identity Bollywood Faces dataset with ArcFace loss. Produces `mobilefacenet_bollywood_scratch.tflite`.  
**Expected runtime:** ~40–80 minutes  
**GPU required:** **YES — T4 ×2**  
**Run in parallel with:** Notebook 04a (fine-tune on the second Kaggle account)

## Strategy

1. Skip the fragile ONNX→Keras conversion entirely. Build MobileFaceNet architecture inline (~100 LOC) — inverted residual blocks, PReLU, global depthwise conv, 512-D embedding output.
2. Pre-crop faces with MediaPipe (same pipeline as 04a; cached to `/kaggle/working/crops/`).
3. ArcFace head on top (100 classes, margin=0.5, scale=64).
4. Train at **higher LR** (1e-3) than the fine-tune notebook — we're learning from zero, not nudging existing weights.
5. **EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)** + **ModelCheckpoint(save_best_only=True)** + ReduceLROnPlateau.
6. Drop the ArcFace head, export the backbone → TFLite.

## Honest expectations

Training a face recognition model from scratch on 12k images is a small budget. Real ArcFace models train on millions of identities. Expectations:

- The model will learn to classify the 100 Bollywood celebs reasonably well (val classification accuracy 60–85%).
- Embedding quality for **arbitrary** faces (not in the 100 celebs) will be weaker than InsightFace's WebFace600K-trained baseline.
- This is mainly an **architecture-validation** + **fallback** notebook. The real win comes if 04a (fine-tune) fails to load the pretrained weights — this notebook still produces a working `.tflite`.
- Notebook 05 will compare both 04a and 04b outputs against the InsightFace baseline via pair verification on Bollywood + LFW, and ship whichever has the lowest EER. The InsightFace baseline may very well win.

## What to send back to Claude

Paste the literal output of cells 5, 7, 9, 11, 13, 15, 17, 19, 23, 25, 27 (discovery, gathering, cropping, split, pipeline, backbone summary, training model, training, TFLite conversion, sanity, summary). If anything fails, paste the **full traceback**.

## 1 — Config

Higher LR than 04a (1e-3 vs 1e-4) — random init can absorb a larger step size.

In [ ]:
STRATEGY               = "from_scratch"
NUM_CLASSES            = 100
EPOCHS_MAX             = 20
BATCH_SIZE             = 64
IMAGE_SIZE             = (112, 112)
EMBEDDING_DIM          = 512
LR                     = 1e-3              # higher for from-scratch
ARCFACE_SCALE          = 64.0
ARCFACE_MARGIN         = 0.5
VAL_SPLIT              = 0.1               # per-celeb
SEED                   = 42
TFLITE_FILENAME        = "mobilefacenet_bollywood_scratch.tflite"

WORK_DIR    = "/kaggle/working"
MODELS_OUT  = f"{WORK_DIR}/models"
REPORTS_OUT = f"{WORK_DIR}/reports"
CROPS_DIR   = f"{WORK_DIR}/crops"

import os
for d in (MODELS_OUT, REPORTS_OUT, CROPS_DIR):
    os.makedirs(d, exist_ok=True)
print(f"Strategy: {STRATEGY}")
print(f"Config locked. LR={LR}, epochs_max={EPOCHS_MAX}, batch={BATCH_SIZE}")

## 2 — Clone repo + install deps

We need mediapipe for face cropping. We do **not** need onnx2tf here — no ONNX conversion.

In [ ]:
import subprocess, sys

REPO_URL = "https://github.com/MalayM09/nhai.git"
REPO_DIR = os.path.join(WORK_DIR, "nhai")
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("CWD:", os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

print("\nInstalling deps (mediapipe, sklearn)…")
!pip install -q mediapipe scikit-learn

## 3 — Discover Bollywood dataset

Three sub-folders (`bollywood_celeb_faces_0`, `bollywood_celeb_faces_1`, `bollywood_celeb_faces2`) under the dataset root.

In [ ]:
import glob

print("/kaggle/input contents:")
if os.path.isdir("/kaggle/input"):
    for d in sorted(os.listdir("/kaggle/input")):
        print(f"  {d}")

DATA_ROOT_CANDIDATES = [
    "/kaggle/input/100-bollywood-celebrity-faces",
    "/kaggle/input/datasets/havingfun/100-bollywood-celebrity-faces",
]
DATA_ROOT = next((c for c in DATA_ROOT_CANDIDATES if os.path.isdir(c)), None)
if DATA_ROOT is None:
    hits = glob.glob("/kaggle/input/**/bollywood_celeb_faces*", recursive=True)
    if hits:
        DATA_ROOT = os.path.dirname(hits[0])
assert DATA_ROOT, "Could not locate Bollywood dataset — paste /kaggle/input/ tree to Claude"
print(f"\nDATA_ROOT: {DATA_ROOT}")

SPLIT_FOLDERS = sorted([os.path.join(DATA_ROOT, d) for d in os.listdir(DATA_ROOT)
                       if os.path.isdir(os.path.join(DATA_ROOT, d)) and d.startswith("bollywood_celeb_faces")])
print(f"\nSplit folders ({len(SPLIT_FOLDERS)}):")
for f in SPLIT_FOLDERS:
    n_celebs = len([d for d in os.listdir(f) if os.path.isdir(os.path.join(f, d))])
    print(f"  {f}  ({n_celebs} celebs)")

## 4 — Walk dataset, gather images per celebrity

In [ ]:
from collections import defaultdict

IMG_EXTS = (".jpg", ".jpeg", ".png")
celeb_to_imgs = defaultdict(list)

for split_folder in SPLIT_FOLDERS:
    for celeb_dir in sorted(os.listdir(split_folder)):
        celeb_path = os.path.join(split_folder, celeb_dir)
        if not os.path.isdir(celeb_path):
            continue
        for f in os.listdir(celeb_path):
            if f.lower().endswith(IMG_EXTS):
                celeb_to_imgs[celeb_dir].append(os.path.join(celeb_path, f))

celebs = sorted(celeb_to_imgs.keys())
celeb_to_idx = {c: i for i, c in enumerate(celebs)}

counts = [len(celeb_to_imgs[c]) for c in celebs]
print(f"Total celebs: {len(celebs)}")
print(f"Total images: {sum(counts)}")
print(f"Min/max/median per celeb: {min(counts)}/{max(counts)}/{sorted(counts)[len(counts)//2]}")

if NUM_CLASSES != len(celebs):
    print(f"\nNOTE: NUM_CLASSES in config ({NUM_CLASSES}) != actual celeb count ({len(celebs)}). Adjusting.")
    NUM_CLASSES = len(celebs)

## 5 — Pre-compute face crops with MediaPipe

Identical pipeline to 04a. Largest detection → 20% margin → 112×112. Centre-crop fallback if no detection.

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from tqdm.auto import tqdm

mp_fd = mp.solutions.face_detection

def detect_and_crop(img_path, detector, out_size=112, margin=0.2):
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        return None
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    results = detector.process(img_rgb)
    if results.detections:
        det = max(results.detections,
                  key=lambda d: d.location_data.relative_bounding_box.width *
                                d.location_data.relative_bounding_box.height)
        bb = det.location_data.relative_bounding_box
        x = int(bb.xmin * w); y = int(bb.ymin * h)
        bw = int(bb.width * w); bh = int(bb.height * h)
        x = max(0, x - int(bw * margin))
        y = max(0, y - int(bh * margin))
        bw = min(w - x, int(bw * (1 + 2 * margin)))
        bh = min(h - y, int(bh * (1 + 2 * margin)))
        face = img_rgb[y:y+bh, x:x+bw]
        used_detection = True
    else:
        s = min(h, w)
        face = img_rgb[(h-s)//2:(h+s)//2, (w-s)//2:(w+s)//2]
        used_detection = False
    return cv2.resize(face, (out_size, out_size), interpolation=cv2.INTER_LINEAR), used_detection

detector = mp_fd.FaceDetection(model_selection=0, min_detection_confidence=0.3)
n_detected = n_fallback = n_failed = 0
crop_paths_per_celeb = defaultdict(list)

all_pairs = [(c, p) for c in celebs for p in celeb_to_imgs[c]]
for celeb, src in tqdm(all_pairs, desc="cropping"):
    out_dir = os.path.join(CROPS_DIR, celeb)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, os.path.splitext(os.path.basename(src))[0] + ".jpg")
    if os.path.isfile(out_path):
        crop_paths_per_celeb[celeb].append(out_path)
        continue
    result = detect_and_crop(src, detector)
    if result is None:
        n_failed += 1
        continue
    face_rgb, used = result
    cv2.imwrite(out_path, cv2.cvtColor(face_rgb, cv2.COLOR_RGB2BGR))
    crop_paths_per_celeb[celeb].append(out_path)
    n_detected += int(used)
    n_fallback += int(not used)

detector.close()
total_crops = sum(len(v) for v in crop_paths_per_celeb.values())
print(f"\nCropping complete:")
print(f"  Total crops produced: {total_crops}")
print(f"  MediaPipe detections: {n_detected}")
print(f"  Centre-crop fallback: {n_fallback}")
print(f"  Failed (skipped):     {n_failed}")
print(f"  Detection rate:       {n_detected / max(1, n_detected + n_fallback):.1%}")

## 6 — Train/val split (stratified per celebrity)

In [ ]:
import random

rng = random.Random(SEED)
train_files, train_labels = [], []
val_files,   val_labels   = [], []
val_files_per_celeb = {}

for celeb in celebs:
    imgs = crop_paths_per_celeb[celeb][:]
    rng.shuffle(imgs)
    n_val = max(1, int(len(imgs) * VAL_SPLIT))
    val_part = imgs[:n_val]; train_part = imgs[n_val:]
    label = celeb_to_idx[celeb]
    train_files.extend(train_part); train_labels.extend([label] * len(train_part))
    val_files.extend(val_part);     val_labels.extend([label] * len(val_part))
    val_files_per_celeb[celeb] = val_part

print(f"Train: {len(train_files)}  ({len(train_labels)} labels)")
print(f"Val:   {len(val_files)}    ({len(val_labels)} labels)")
print(f"Per-celeb val min/max: {min(len(v) for v in val_files_per_celeb.values())}/{max(len(v) for v in val_files_per_celeb.values())}")

## 7 — `tf.data` pipeline

Same as 04a: read pre-cropped JPG → normalize to `[-1, 1]` → cache → shuffle → augment (mild) → batch as `((image, label), label)`.

In [ ]:
import tensorflow as tf

AUTOTUNE = tf.data.AUTOTUNE

def decode_normalize(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_jpeg(img, channels=3)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMAGE_SIZE)
    img = (tf.cast(img, tf.float32) - 127.5) / 127.5
    return img, label

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.1)
    img = tf.image.random_contrast(img, lower=0.9, upper=1.1)
    img = tf.clip_by_value(img, -1.0, 1.0)
    return img, label

def make_ds(files, labels, training):
    combined = list(zip(files, labels))
    random.Random(SEED + (0 if training else 1)).shuffle(combined)
    files, labels = [list(x) for x in zip(*combined)]
    ds = tf.data.Dataset.from_tensor_slices((files, labels))
    ds = ds.map(decode_normalize, num_parallel_calls=AUTOTUNE)
    ds = ds.cache()
    if training:
        ds = ds.shuffle(buffer_size=8192, seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda img, lbl: ((img, lbl), lbl), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(train_files, train_labels, training=True)
val_ds   = make_ds(val_files,   val_labels,   training=False)

for (img_b, lbl_b), lbl_t in train_ds.take(1):
    print(f"image batch: shape={img_b.shape} range=[{tf.reduce_min(img_b):.3f}, {tf.reduce_max(img_b):.3f}]")
    print(f"label batch: shape={lbl_b.shape} sample={lbl_b[:8].numpy().tolist()}")
    print(f"unique classes in this batch: {len(set(lbl_b.numpy().tolist()))}")
print(f"\nGPU(s): {tf.config.list_physical_devices('GPU') or 'NONE — switch accelerator to T4 ×2'}")

## 8 — Build MobileFaceNet architecture from scratch

Compact implementation following the original paper. Stages:

1. Stem (conv 3×3 stride 2 → depthwise 3×3) → 56×56
2. Bottleneck block ×5 (channels=64) → 28×28
3. Bottleneck block ×7 (channels=128) → 14×14
4. Bottleneck block ×3 (channels=128) → 7×7
5. Conv 1×1 → 7×7×512
6. Global depthwise conv (7×7 kernel) → 1×1×512
7. Conv 1×1 → 1×1×512
8. Flatten + BN → 512-D embedding

Activation: PReLU (the original choice; more expressive than ReLU). All convs use Keras 3 layers — no raw `tf.*` on KerasTensors.

In [ ]:
from tensorflow.keras import layers as L
import keras

def conv_bn_prelu(x, filters, kernel_size=3, strides=1):
    x = L.Conv2D(filters, kernel_size, strides=strides, padding="same", use_bias=False)(x)
    x = L.BatchNormalization()(x)
    return L.PReLU(shared_axes=[1, 2])(x)

def dw_bn_prelu(x, kernel_size=3, strides=1):
    x = L.DepthwiseConv2D(kernel_size, strides=strides, padding="same", use_bias=False)(x)
    x = L.BatchNormalization()(x)
    return L.PReLU(shared_axes=[1, 2])(x)

def inverted_residual(x, filters_out, expansion=2, strides=1):
    filters_in = x.shape[-1]
    expanded = filters_in * expansion
    y = conv_bn_prelu(x, expanded, 1, 1)
    y = dw_bn_prelu(y, 3, strides)
    y = L.Conv2D(filters_out, 1, padding="same", use_bias=False)(y)
    y = L.BatchNormalization()(y)
    if strides == 1 and filters_in == filters_out:
        y = L.Add()([y, x])
    return y

def build_mobilefacenet(input_shape=(112, 112, 3), embedding_dim=512):
    inp = L.Input(shape=input_shape, name="image")
    
    # Stem: 112x112x3 -> 56x56x64
    x = conv_bn_prelu(inp, 64, 3, strides=2)
    x = dw_bn_prelu(x, 3, 1)
    
    # Stage 1: 56x56x64 -> 28x28x64
    x = inverted_residual(x, 64, expansion=2, strides=2)
    for _ in range(4):
        x = inverted_residual(x, 64, expansion=2, strides=1)
    
    # Stage 2: 28x28x64 -> 14x14x128
    x = inverted_residual(x, 128, expansion=4, strides=2)
    for _ in range(6):
        x = inverted_residual(x, 128, expansion=2, strides=1)
    
    # Stage 3: 14x14x128 -> 7x7x128
    x = inverted_residual(x, 128, expansion=4, strides=2)
    for _ in range(2):
        x = inverted_residual(x, 128, expansion=2, strides=1)
    
    # Final 1x1 conv: 7x7x128 -> 7x7x512
    x = conv_bn_prelu(x, 512, 1, 1)
    
    # Global depthwise conv (kernel = 7x7 = feature map size)
    x = L.DepthwiseConv2D(7, strides=1, padding="valid", use_bias=False)(x)
    x = L.BatchNormalization()(x)
    
    # 1x1 conv to embedding dim + flatten + BN
    x = L.Conv2D(embedding_dim, 1, padding="valid", use_bias=False)(x)
    x = L.Flatten()(x)
    embedding = L.BatchNormalization(name="embedding")(x)
    
    return tf.keras.Model(inp, embedding, name="mobilefacenet_scratch")

backbone = build_mobilefacenet(input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3), embedding_dim=EMBEDDING_DIM)
backbone.summary(line_length=120)
n_params = backbone.count_params()
print(f"\nTotal params: {n_params:,}")
print(f"FP32 size estimate: {n_params * 4 / 1024 / 1024:.2f} MB")

# Sanity forward
test_in = tf.random.uniform((1, 112, 112, 3), -1.0, 1.0)
test_out = backbone(test_in)
print(f"\nSanity forward: input {test_in.shape} -> output {test_out.shape}")
print(f"Output norm (random init): {float(tf.norm(test_out)):.3f}")

## 9 — ArcFace head + training model

Identical to 04a's head.

In [ ]:
class ArcFaceHead(tf.keras.layers.Layer):
    def __init__(self, num_classes, scale=64.0, margin=0.5, **kwargs):
        super().__init__(**kwargs)
        self.num_classes = num_classes
        self.scale = scale
        self.margin = margin
    def build(self, input_shape):
        emb_dim = input_shape[0][-1]
        self.kernel = self.add_weight(
            name="kernel", shape=(emb_dim, self.num_classes),
            initializer="glorot_uniform", trainable=True)
        super().build(input_shape)
    def call(self, inputs):
        embeddings, labels = inputs
        eps = 1e-9
        e_n = embeddings / (keras.ops.sqrt(keras.ops.sum(keras.ops.square(embeddings), axis=1, keepdims=True)) + eps)
        k_n = self.kernel    / (keras.ops.sqrt(keras.ops.sum(keras.ops.square(self.kernel),    axis=0, keepdims=True)) + eps)
        cos_t = keras.ops.matmul(e_n, k_n)
        cos_t = keras.ops.clip(cos_t, -1.0 + 1e-7, 1.0 - 1e-7)
        sin_t = keras.ops.sqrt(1.0 - keras.ops.square(cos_t))
        cm = keras.ops.cos(self.margin); sm = keras.ops.sin(self.margin)
        cos_tm = cos_t * cm - sin_t * sm
        mask = keras.ops.one_hot(keras.ops.cast(labels, "int32"), self.num_classes)
        logits = keras.ops.where(mask > 0.5, cos_tm, cos_t)
        return logits * self.scale
    def get_config(self):
        return {**super().get_config(),
                "num_classes": self.num_classes,
                "scale": self.scale, "margin": self.margin}

img_input   = L.Input(shape=(112, 112, 3), name="image")
label_input = L.Input(shape=(), dtype="int32", name="label")

embeddings = backbone(img_input)
logits     = ArcFaceHead(num_classes=NUM_CLASSES,
                         scale=ARCFACE_SCALE,
                         margin=ARCFACE_MARGIN, name="arcface")([embeddings, label_input])

training_model = tf.keras.Model([img_input, label_input], logits, name="training_model")

training_model.compile(
    optimizer=tf.keras.optimizers.Adam(LR),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["sparse_categorical_accuracy"],
)
training_model.summary(line_length=120)

## 10 — Train

Up to 20 epochs with early stopping. Same callback config as 04a.

In [ ]:
import json

BEST_CKPT = f"{WORK_DIR}/best_scratch.keras"
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=BEST_CKPT, monitor="val_loss", save_best_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1),
]

history = training_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_MAX,
    callbacks=callbacks,
    verbose=2,
)

HIST_PATH = f"{REPORTS_OUT}/mobilefacenet_scratch_training_history.json"
hist_data = {k: [float(v) for v in vals] for k, vals in history.history.items()}
hist_data["strategy"]      = STRATEGY
hist_data["epochs_actual"] = len(history.history.get("loss", []))
hist_data["epochs_max"]    = EPOCHS_MAX
hist_data["lr"]            = LR
hist_data["num_classes"]   = NUM_CLASSES
hist_data["best_val_loss"] = float(min(history.history.get("val_loss", [float("inf")])))
with open(HIST_PATH, "w") as f:
    json.dump(hist_data, f, indent=2)
print(f"\nHistory saved -> {HIST_PATH}")

## 11 — Export backbone as TFLite

In [ ]:
inf_inp = L.Input(shape=(112, 112, 3))
inf_out = backbone(inf_inp)
inference_model = tf.keras.Model(inf_inp, inf_out, name="mfn_scratch_inference")

TFLITE_PATH = f"{MODELS_OUT}/{TFLITE_FILENAME}"
converter = tf.lite.TFLiteConverter.from_keras_model(inference_model)
tflite_bytes = converter.convert()
with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_bytes)
print(f"TFLite saved -> {TFLITE_PATH}  ({os.path.getsize(TFLITE_PATH)/1024/1024:.2f} MB)")

it = tf.lite.Interpreter(model_path=TFLITE_PATH)
it.allocate_tensors()
ish = it.get_input_details()[0]["shape"].tolist()
osh = it.get_output_details()[0]["shape"].tolist()
print(f"Input:  {ish}"); print(f"Output: {osh}")
assert ish == [1, 112, 112, 3], f"input shape mismatch: {ish}"
assert osh == [1, 512],        f"output shape mismatch: {osh}"

## 12 — Sanity check

Same-celeb pairs should have lower cosine distance than cross-celeb pairs. If the model trained well, expect:

- Same-celeb: 0.3–0.5 cosine distance
- Cross-celeb: 0.7–1.2 cosine distance

If same and cross overlap a lot, the from-scratch training didn't produce useful embeddings (likely the case with only 12k images). That's why Notebook 05 compares against the InsightFace baseline.

In [ ]:
rng2 = random.Random(SEED + 100)
test_celebs = rng2.sample([c for c in celebs if len(val_files_per_celeb[c]) >= 2], 3)

def tflite_embed(it, img_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (112, 112)).astype(np.float32)
    img = (img - 127.5) / 127.5
    img = img[None].astype(it.get_input_details()[0]["dtype"])
    it.set_tensor(it.get_input_details()[0]["index"], img)
    it.invoke()
    e = it.get_tensor(it.get_output_details()[0]["index"])[0]
    return e / (np.linalg.norm(e) + 1e-9)

it = tf.lite.Interpreter(model_path=TFLITE_PATH)
it.allocate_tensors()

embs = {c: [tflite_embed(it, p) for p in val_files_per_celeb[c][:3]] for c in test_celebs}

print("Same-celeb pairwise cosine distances (lower = more similar; aim < 0.5):")
for c, es in embs.items():
    if len(es) < 2: continue
    for i in range(len(es)):
        for j in range(i+1, len(es)):
            d = 1.0 - float(np.dot(es[i], es[j]))
            print(f"  {c:30s} pair ({i},{j}): {d:.4f}")

print("\nCross-celeb pairwise cosine distances (aim > 0.7):")
for i, c1 in enumerate(test_celebs):
    for j, c2 in enumerate(test_celebs):
        if i >= j: continue
        for e1 in embs[c1][:1]:
            for e2 in embs[c2][:1]:
                d = 1.0 - float(np.dot(e1, e2))
                print(f"  {c1[:20]:20s} vs {c2[:20]:20s}: {d:.4f}")

## 13 — Plots + final summary

In [ ]:
import matplotlib.pyplot as plt

h = history.history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (key, ylabel) in zip(axes, [("loss", "Loss"), ("sparse_categorical_accuracy", "Accuracy")]):
    if key in h and f"val_{key}" in h:
        ax.plot(h[key], label=f"train {key}")
        ax.plot(h[f"val_{key}"], label=f"val {key}")
        ax.set_xlabel("epoch"); ax.set_ylabel(ylabel); ax.legend(); ax.grid(alpha=0.3)
fig.suptitle(f"MobileFaceNet FROM-SCRATCH on Bollywood Faces (best val_loss={hist_data['best_val_loss']:.4f})")
fig.tight_layout()
PLOT_PATH = f"{REPORTS_OUT}/mobilefacenet_scratch_training_curves.png"
fig.savefig(PLOT_PATH, dpi=120, bbox_inches="tight")
plt.show()
print(f"Plot saved -> {PLOT_PATH}")

print("\n" + "=" * 78)
print(f"Final summary — Notebook 04b ({STRATEGY})")
print("=" * 78)
print(f"Total celebs (classes):  {NUM_CLASSES}")
print(f"Train samples:           {len(train_files):,}")
print(f"Val samples:             {len(val_files):,}")
print(f"Backbone params:         {backbone.count_params():,}")
print(f"Epochs ran:              {hist_data['epochs_actual']} of {EPOCHS_MAX} max")
print(f"Final train acc:         {h['sparse_categorical_accuracy'][-1]:.4f}")
print(f"Final val   acc:         {h['val_sparse_categorical_accuracy'][-1]:.4f}")
print(f"Best  val   loss:        {hist_data['best_val_loss']:.4f}")
print()
print(f"TFLite file:             {TFLITE_PATH}")
print(f"TFLite size:             {os.path.getsize(TFLITE_PATH)/1024/1024:.2f} MB")
print(f"History JSON:            {HIST_PATH}")
print(f"Curves PNG:              {PLOT_PATH}")

## What to send back to Claude

Paste literal output of cells 5, 7, 9, 11, 13, 15, 17, 19, 23, 25, 27 (discovery, gathering, cropping, split, pipeline, backbone summary, training model, training, TFLite conversion, sanity, final summary). If anything fails, paste the **full traceback**.

## After the run — committing the artifacts

Download to laptop into `kaggle_downloads/04b_mobilefacenet_scratch/`:

- `/kaggle/working/models/mobilefacenet_bollywood_scratch.tflite`
- `/kaggle/working/reports/mobilefacenet_scratch_training_history.json`
- `/kaggle/working/reports/mobilefacenet_scratch_training_curves.png`

Tell Claude when both 04a and 04b artifacts are downloaded. **Don't replace `mobile_app/assets/models/mobilefacenet.tflite` yet** — Notebook 05 evaluates both fine-tuned variants against the InsightFace baseline via pair verification, picks the EER winner, and only then do we ship.